# Phase 4 — Speaker Verification (demonstration with an UNTRAINED encoder)

**Question:** *are these two recordings from the same speaker?*

```
AUDIO A -> SpeakerEncoder -> Embedding A ─┐
                                          ├─> cosine similarity -> >= threshold ? -> MATCH / NOT MATCH
AUDIO B -> SpeakerEncoder -> Embedding B ─┘
```

> **DEMONSTRATION ONLY — the encoder is not trained.** Every decision below comes from random weights.
> With one speaker and three recordings we also cannot measure how often impostors would be accepted.

## 1. Device: CUDA if available, otherwise CPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Clone the repository and install requirements

In [ ]:
import os
if not os.path.exists("/content/Voice_engine"):
    !git clone https://github.com/8919134556/Voice_engine.git /content/Voice_engine
%cd /content/Voice_engine
!git pull
!pip install -q -r requirements.txt

## 3. Private dataset
**Option A — Google Drive (recommended):** `MyDrive/voice_engine/dataset/speaker_01/audio_00X.wav`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATASET_DIR = "/content/drive/MyDrive/voice_engine/dataset"
!ls "{DATASET_DIR}/speaker_01"

**Option B — upload for this session only** (skip if you used Drive).

In [ ]:
# Run ONLY if you are not using Google Drive.
import os, shutil
from google.colab import files
DATASET_DIR = "/content/Voice_engine/dataset"
os.makedirs(f"{DATASET_DIR}/speaker_01", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"{DATASET_DIR}/speaker_01/{name}")
!ls "{DATASET_DIR}/speaker_01"

## 4. Compare two files

In [ ]:
!python scripts/verify_speaker.py "{DATASET_DIR}/speaker_01/audio_001.wav" "{DATASET_DIR}/speaker_01/audio_002.wav"

In [ ]:
# Another pair, with a different threshold
!python scripts/verify_speaker.py "{DATASET_DIR}/speaker_01/audio_002.wav" "{DATASET_DIR}/speaker_01/audio_003.wav" --threshold 0.9

## 5. Score every pair in the dataset

In [ ]:
!python scripts/evaluate_pairs.py --dataset "{DATASET_DIR}"

In [ ]:
from IPython.display import Image
Image("outputs/phase4/pair_scores.png")

---
# Step by step with `SpeakerVerifier`

In [ ]:
from pathlib import Path
from src.audio.loader import list_audio_files
from src.verification.verifier import create_verifier, UNTRAINED_WARNING

verifier = create_verifier(seed=0, device=device)   # same untrained weights as Phase 3
print("Trained encoder?", verifier.trained, "->", UNTRAINED_WARNING)

wav_files = list_audio_files(Path(DATASET_DIR) / "speaker_01")
emb = {p.stem: verifier.generate_embedding(p) for p in wav_files}
for name, e in emb.items():
    print(name, e.shape)

In [ ]:
# Genuine pairs (same speaker). DEMONSTRATION ONLY.
from itertools import combinations
for a, b in combinations(emb, 2):
    result = verifier.verify(emb[a], emb[b], threshold=0.70)
    print(f"{a} vs {b}   similarity {result.similarity:.4f}   decision {result.decision}")

### The threshold decides — and it is arbitrary right now
Same similarity scores, different thresholds, different decisions. Without impostor pairs
there is no data to tell us which threshold is right.

In [ ]:
a, b = list(emb)[:2]
for threshold in [0.5, 0.7, 0.9, 0.99, 0.999, 0.9999]:
    print(f"threshold {threshold:<7} -> {verifier.verify(emb[a], emb[b], threshold).decision}")

---
# Illustration: FAR, FRR and EER

**The scores in the next cell are MADE-UP numbers** drawn from two bell curves.
They are NOT from your recordings and NOT from our encoder. They only show what a *trained*
system's genuine and impostor score distributions might look like, and how the threshold trades
false accepts against false rejects.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
genuine_scores = np.clip(rng.normal(0.75, 0.08, 500), -1, 1)    # HYPOTHETICAL same-speaker scores
impostor_scores = np.clip(rng.normal(0.35, 0.12, 500), -1, 1)   # HYPOTHETICAL different-speaker scores

thresholds = np.linspace(0, 1, 501)
far = np.array([(impostor_scores >= t).mean() for t in thresholds])  # impostors wrongly accepted
frr = np.array([(genuine_scores < t).mean() for t in thresholds])    # genuine speakers wrongly rejected
i = np.argmin(np.abs(far - frr))
print(f"EER ~ {(far[i] + frr[i]) / 2:.1%} at threshold {thresholds[i]:.2f}  (hypothetical data!)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(genuine_scores, bins=40, alpha=0.6, label="genuine pairs")
ax1.hist(impostor_scores, bins=40, alpha=0.6, label="impostor pairs")
ax1.axvline(thresholds[i], color="red", linestyle="--", label="EER threshold")
ax1.set_title("HYPOTHETICAL score distributions"); ax1.set_xlabel("cosine similarity"); ax1.legend()
ax2.plot(thresholds, far, label="FAR (false accepts)")
ax2.plot(thresholds, frr, label="FRR (false rejects)")
ax2.scatter([thresholds[i]], [far[i]], color="red", zorder=3, label="EER point")
ax2.set_title("Moving the threshold trades FAR against FRR"); ax2.set_xlabel("threshold"); ax2.legend()
plt.tight_layout()

### Privacy
Recordings and embeddings are personal (biometric) data. They are git-ignored; keep them local or in private Drive.